![img](https://github.com/JuliaLang/julia/raw/master/doc/src/assets/logo.svg)![img](https://avatars.githubusercontent.com/u/7346142?s=200&v=4)

# Installation and Setup

JuliaGPU packages are easy to install: Just do `Pkg.add("CUDA")` to install the CUDA.jl package, which provides bindings to NVIDIA's CUDA. CUDA.jl provides all of the compiler and runtime logic needed to program NVIDIA GPUs; the only thing you need to provide is a functional NVIDIA driver (which most HPC systems already have installed and configured), but you don't need to install the CUDA toolkit! CUDA.jl downloads one if it's not already available on your system (again, HPC systems usually have this provided for you):

In [54]:
# This can take a little while to download and compile, so just be patient
import Pkg
Pkg.add("CUDA")

   Resolving package versions...
  No Changes to `/global/u1/o/omairyrm/.julia/environments/v1.10/Project.toml`
  No Changes to `/global/u1/o/omairyrm/.julia/environments/v1.10/Manifest.toml`


We're going to be running some small benchmarks in this notebook, so let's also grab Julia's BenchmarkTools.jl while we're at it:

In [ ]:
import Pkg
Pkg.add("BenchmarkTools")

   Resolving package versions...
  No Changes to `/global/u1/o/omairyrm/.julia/environments/v1.10/Project.toml`
  No Changes to `/global/u1/o/omairyrm/.julia/environments/v1.10/Manifest.toml`


And now we import these packages, along with the built-in LinearAlgebra standard library:

In [3]:
using CUDA
using BenchmarkTools

using LinearAlgebra

Now, GPU vendor libraries can be difficult, so CUDA.jl provides a convenient way to check if everything is setup correctly, the `CUDA.versioninfo()` function. Like Julia's `Base.versioninfo()`, this will print some information on the available hardware and loaded libraries:

In [4]:
CUDA.versioninfo()

CUDA runtime 12.4, local installation
CUDA driver 12.8
NVIDIA driver 550.127.8

CUDA libraries: 
- CUBLAS: 12.4.2
- CURAND: 10.3.5
- CUFFT: 11.2.1
11.6.1LVER: 
- CUSPARSE: 12.3.1
- CUPTI: 2024.1.1 (API 22.0.0)
- NVML: 12.0.0+550.127.8

Julia packages: 
- CUDA: 5.7.3
- CUDA_Driver_jll: 0.12.1+1
- CUDA_Runtime_jll: 0.16.1+0
- CUDA_Runtime_Discovery: 0.3.5

Toolchain:
- Julia: 1.10.9
- LLVM: 15.0.7

Preferences:
- CUDA_Runtime_jll.version: 12.4
- CUDA_Runtime_jll.local: true

1 device:
  0: NVIDIA A100-SXM4-40GB (sm_80, 39.377 GiB / 40.000 GiB available)


It's always good practice to check this at least once on a new system or when you mess with `module`s loaded, just to ensure that everything is connected and happy!

# Array Programming on GPU

The goal of CUDA.jl is to make GPU arrays behave just like CPU arrays (Array)—following the same API as much as possible.


## Creating CuArrays

You can create uninitialized GPU arrays just like with Array:

In [33]:
CuArray{Float32,2}(undef, 2, 2)

2×2 CuArray{Float32, 2, CUDA.DeviceMemory}:
 0.0    1.0f-45
 1.875  0.0

You can also create GPU arrays from existing Julia arrays:

In [34]:
a = CuArray([1 2 3])

1×3 CuArray{Int64, 2, CUDA.DeviceMemory}:
 1  2  3

In [35]:
b = Array([1, 2, 3, 4])
a = CuArray(b)

4-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 1
 2
 3
 4

Move data to CPU:

In [36]:
b = Array(a)

4-element Vector{Int64}:
 1
 2
 3
 4

similar works just like on the CPU, creating arrays with the same shape and element type:

In [38]:
similar(a)

4-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 4697255047510937792
                   1
                   4
                   4

The goal of CUDA.jl is to maintain API compatibility with Base.Array, allowing you to write code that works seamlessly on both CPU and GPU with minimal changes.


Just like Base.Array, CUDA.jl provides familiar constructors to create and initialize GPU arrays:

In [40]:
CUDA.ones(2)

2-element CuArray{Float32, 1, CUDA.DeviceMemory}:
 1.0
 1.0

In [41]:
 CUDA.zeros(Float32, 2)

2-element CuArray{Float32, 1, CUDA.DeviceMemory}:
 0.0
 0.0

In [42]:
CUDA.fill(42, (3,4))

3×4 CuArray{Int64, 2, CUDA.DeviceMemory}:
 42  42  42  42
 42  42  42  42
 42  42  42  42

You can also generate random arrays directly on the GPU:

In [43]:
CUDA.rand(2, 2)

2×2 CuArray{Float32, 2, CUDA.DeviceMemory}:
 0.562833   0.961987
 0.0634769  0.821073

Or preallocate and fill in-place:

In [44]:
a = CuArray{Float32}(undef, (2,2))
rand!(a)

2×2 CuArray{Float32, 2, CUDA.DeviceMemory}:
 0.290407  0.738942
 0.813068  0.799476


# Linear Algebra on the GPU

CUDA.jl seamlessly integrates with LinearAlgebra and underlying libraries like CUBLAS, CUSOLVER, and CUFFT.

In [45]:
a * a

2×2 CuArray{Float32, 2, CUDA.DeviceMemory}:
 0.685146  0.80536
 0.886149  1.23997

In [46]:
LinearAlgebra.qr!(a) # QR factorization (CUSOLVER)

QR{Float32, CuArray{Float32, 2, CUDA.DeviceMemory}, CuArray{Float32, 1, CUDA.DeviceMemory}}
Q factor: 2×2 LinearAlgebra.QRPackedQ{Float32, CuArray{Float32, 2, CUDA.DeviceMemory}, CuArray{Float32, 1, CUDA.DeviceMemory}}
R factor:
2×2 CuArray{Float32, 2, CUDA.DeviceMemory}:
 -0.863375  -1.00144
  0.0       -0.426972

In [47]:
 CUFFT.plan_fft(a) * a

2×2 CuArray{ComplexF32, 2, CUDA.DeviceMemory}:
 -1.58709+0.0im  1.26974+0.0im
 -2.14255+0.0im  -0.9936+0.0im

# High-Level Functional Programming

GPU arrays support familiar functional operations:

In [49]:
a = CuArray([1 2 3])
b = CuArray([4 5 6])

1×3 CuArray{Int64, 2, CUDA.DeviceMemory}:
 4  5  6

In [50]:
map(a) do x
    x + 1
end

1×3 CuArray{Int64, 2, CUDA.DeviceMemory}:
 2  3  4

Reduction and accumulation:

In [51]:
reduce(+, a)

6

In [52]:
accumulate(+, b; dims=2)

1×3 CuArray{Int64, 2, CUDA.DeviceMemory}:
 4  9  15

In [53]:
findfirst(isequal(2), a)

CartesianIndex(1, 2)

# Example: Vector Addition

As a simple example, let's take a look at vector addition. Let's assume you have two vectors $\vec{a}$ and $\vec{b}$ and you want to add them elementwise. You can do this in many ways in Julia:
1. simple for loop on a CPU
2. julia array add (+) on a CPU or GPU
3. GPU kernel programming in CUDA (or KernelAbstractions using CUDA as backend - we'll see this soon!)

In [5]:
# define our input a, b vectors, and output c vector in CPU RAM
vector_size = 1024
a = rand(1:4, vector_size)
b = rand(1:4, vector_size)
c = zeros(Int, vector_size)

# what's in a?
a

1024-element Vector{Int64}:
 4
 2
 4
 2
 1
 1
 4
 2
 2
 3
 2
 3
 1
 ⋮
 1
 1
 2
 3
 1
 3
 2
 3
 4
 4
 4
 4

Let's write a simple CPU loop to add two vectors in serial:

In [6]:
# Note: the exclamation mark (!) doesn't do anything special
# It's just used to indicate that a function mutates its arguments
function vadd!(a, b, c)
    for i in 1:length(c)
        c[i] = a[i] + b[i]
    end
end
vadd!(a, b, c)
c

1024-element Vector{Int64}:
 7
 3
 5
 4
 5
 4
 7
 3
 3
 6
 3
 7
 4
 ⋮
 4
 3
 4
 4
 2
 5
 5
 4
 7
 5
 6
 5

Thankfully, Julia has a ton of built-in array operations, so we don't actually need to implement this ourselves. Julia's vector add (+) operation works exactly as you'd expect:

In [7]:
c = a + b

# Note that, unlike `vadd!`, the above operation allocates a new `c` as the output vector - this is important to remember.

1024-element Vector{Int64}:
 7
 3
 5
 4
 5
 4
 7
 3
 3
 6
 3
 7
 4
 ⋮
 4
 3
 4
 4
 2
 5
 5
 4
 7
 5
 6
 5

Great! But isn't this a GPU tutorial? Let's get to it!

In [8]:
# We need first to make copies of the a and b vectors on the GPU, and define a new dc empty GPU vector

# The CuArray() function automatically allocates a new GPU array of the same size and shape as the input,
# and copies from the input CPU array to the newly-allocated GPU array
da = CuArray(a)
db = CuArray(b)

# CUDA.zeros takes the desired element type and array size, and automatically allocates and initializes
# a new GPU array with int64 zeros
dc = CUDA.zeros(Int, size(a))

# We can also safely take a look at what's in da, even though it's on the GPU!
# It's a different array type, and that fact is made clear to us:
da

1024-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 4
 2
 4
 2
 1
 1
 4
 2
 2
 3
 2
 3
 1
 ⋮
 1
 1
 2
 3
 1
 3
 2
 3
 4
 4
 4
 4

Now that we know how to allocate on the GPU, let's see how to use this same add (+) operation on the GPU to add two of those vectors using CUDA:

In [9]:
dc = da + db

# We can add GPU vectors using the same `+` operator, thanks to Julia's multiple dispatch!
# Also, like for the CPU add operation, this one also allocates a new GPU array for output `dc`

1024-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 7
 3
 5
 4
 5
 4
 7
 3
 3
 6
 3
 7
 4
 ⋮
 4
 3
 4
 4
 2
 5
 5
 4
 7
 5
 6
 5

Cool, but vector addition is pretty... simple? Let us learn how to write our own GPU kernels with CUDA.jl in pure Julia.

In array operations, CUDA.jl can leverage implicit parallelism (expressed over the array's elements) to automatically execute these operations in parallel on a GPU. When using hand-rolled kernels, it is instead the programmer's responsibility to decide how to effectively assign the available parallel execution resources for the specific operation. Let's see how this is done for vector addition, before moving on to more interesting examples:

In [10]:
function vadd_kernel!(c, a, b)
    # Obtain GPU thread index, which should be mapped to the valid indices of a and b
    i = threadIdx().x
    # Each thread will add its own element to c
    c[i] = a[i] + b[i]

    # GPU kernels don't return anything
    return
end

vadd_kernel! (generic function with 1 method)

At a high level, that's pretty easy, you just need to write a scalar function, just like you'd do if you were writing CUDA C++. Now we just need to launch that function in parallel using the `@cuda` macro, and specify the number of GPU threads with the `threads` keyword argument:

In [11]:
# Launch our `vadd_kernel!` GPU kernel, with our GPU arrays as inputs
@cuda threads=length(da) vadd_kernel!(dc, da, db)

dc

1024-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 7
 3
 5
 4
 5
 4
 7
 3
 3
 6
 3
 7
 4
 ⋮
 4
 3
 4
 4
 2
 5
 5
 4
 7
 5
 6
 5

OK, this is great, and was not a lot of work for us! But to see a downside of this simple approach, let's try to work with bigger vectors, by setting `vector_size` to 10240:

In [12]:
vector_size = 10240
da = CuArray(rand(1:4, vector_size))
db = CuArray(rand(1:4, vector_size))
dc = CUDA.zeros(Int, vector_size)

@cuda threads=length(da) vadd_kernel!(dc, da, db)

LoadError: Number of threads in x-dimension exceeds device limit (10240 > 1024).

Oh no! What is going on here?

GPUs have a limited number of threads they can run on a single streaming multiprocessor (SM) at once, and we just tried to assign too many threads to one SM, which isn't possible:

In [ ]:
# To query the number of threads per block, we can inspect CUDA attributes:
CUDA.attribute(device(), CUDA.DEVICE_ATTRIBUTE_MAX_THREADS_PER_BLOCK)

1024

Since 10240 > 1024, the SM wouldn't have had enough resources to satisfy our request, at least not with the default of a single block per kernel.

Thankfully, GPUs also have multiple SMs, so in theory this should be solvable. To take advantage of more than one SM, we need to run a kernel with multiple blocks, as a single block can only execute on one SM (which has limited resources available, as we saw in the query above). In order to exploit multiple blocks, though, we need to understand how to index into our arrays when our index depends not just on our thread index, but also on our block index and block sizes.

In CUDA.jl, the expression `i = threadIdx().x + (blockIdx().x - 1) * blockDim().x` calculates a unique index for each thread across multiple blocks in a CUDA kernel execution. Here's a breakdown of each component and how they contribute to computing this index:

- `threadIdx().x`: This returns the x-coordinate of the thread within its block. It's the thread's index within the block, starting from 1 (unlike C/C++ CUDA where it starts from 0).

- `blockIdx().x`: This gives the x-coordinate of the block within the grid. It represents the block's index in the grid, also starting from 1.

- `blockDim().x`: This represents the number of threads per block along the x-axis.

In essence, the formula `i = threadIdx().x + (blockIdx().x - 1) * blockDim().x` is used to compute a global index for each thread, regardless of how blocks are sized. It positions the threads linearly across all blocks. Here's what each part does:

- `(blockIdx().x - 1) * blockDim().x`: This part calculates the offset to the start of the current block. Subtracting 1 from `blockIdx().x` makes it zero-based, and then it is multiplied by the number of threads in each block `(blockDim().x)`. This gives the index of the first thread in the current block relative to the entire grid.

- `threadIdx().x`: Adding this to the block offset gives the specific thread's index within the whole grid.

Knowing this, let's now rewrite our kernel to properly handle multiple blocks, using the above global indexing formula:

In [13]:
function vadd_cuda!(c, a, b)
    # Calculate a unique index for each thread across multiple blocks
    i = threadIdx().x + (blockIdx().x - 1) * blockDim().x

    # Ensure that we skip invalid indices, if we over-allocated a few threads
    if i <= length(a)
        c[i] = a[i] + b[i]
    end

    return
end

vadd_cuda! (generic function with 1 method)

Now we can launch our kernel with the maximum number of threads per block (1024), and then divide up our computation across multiple 1024-wide blocks:

In [14]:
@cuda threads=1024 blocks=cld(length(da),1024) vadd_cuda!(dc, da, db) # cld(x, y) is (x / y) with round-up behavior
dc

10240-element CuArray{Int64, 1, CUDA.DeviceMemory}:
 3
 5
 5
 4
 7
 3
 2
 4
 7
 3
 4
 3
 5
 ⋮
 7
 5
 6
 5
 5
 7
 2
 7
 5
 8
 2
 6

Yay! Now that we're thoroughly done with vector addition, let's move on to something a bit heavier:

## Example: Matrix-Matrix Multiplication

Matrix multiplication is a mainstay of all kinds of applications, so we should be able to implement this in Julia with ease. Let's first take a look at doing this on the CPU:

In [15]:
# Allocate our random matrix inputs and zero'd output
matrix_size = 1024
A = rand(matrix_size, matrix_size)
B = rand(matrix_size, matrix_size)
C = zeros(matrix_size, matrix_size)

A

1024×1024 Matrix{Float64}:
 0.660089   0.910209  0.547063  …  0.964938    0.609727   0.677683
 0.326237   0.544675  0.652396     0.185658    0.455686   0.992389
 0.711058   0.152011  0.392105     0.787188    0.487585   0.804315
 0.281303   0.46956   0.137244     0.24401     0.186616   0.558793
 0.825143   0.204056  0.292768     0.905266    0.908755   0.777511
 0.874345   0.225637  0.822056  …  0.686041    0.0494679  0.0804205
 0.602205   0.351695  0.359766     0.161786    0.724337   0.28782
 0.785481   0.703428  0.904731     0.919084    0.24644    0.548605
 0.505204   0.255183  0.161325     0.780454    0.0791342  0.815757
 0.513395   0.700414  0.850081     0.440594    0.760314   0.385778
 0.72773    0.453     0.494087  …  0.98283     0.48709    0.856695
 0.560429   0.139243  0.598645     0.260218    0.471614   0.286441
 0.127678   0.177489  0.296763     0.836852    0.602944   0.708262
 ⋮                              ⋱                         
 0.367329   0.656017  0.306023     0.678939

The three nested loops implementation of matrix multiplication is easy to express on the CPU:

In [16]:
function MatrixMultiplication!(C, A, B)
    for i in 1:size(C, 1)
        for j in 1:size(C, 2)
            C[i, j] = 0
            for k in 1:size(A, 2)
                C[i, j] += A[i, k] * B[k, j]
            end
        end
    end
end


MatrixMultiplication! (generic function with 1 method)

In [17]:
MatrixMultiplication!(C, A, B)
C

1024×1024 Matrix{Float64}:
 259.863  254.538  255.66   246.242  …  261.408  255.862  253.906  260.033
 263.178  264.622  266.778  252.049     268.632  263.909  257.352  271.766
 258.756  254.056  252.085  247.739     258.581  256.31   248.666  260.403
 269.797  265.681  261.253  255.096     272.29   257.799  260.155  274.929
 255.627  251.289  257.524  243.523     256.933  254.892  250.325  261.133
 251.389  252.676  253.779  241.805  …  253.826  251.734  248.212  260.028
 244.499  246.652  245.788  237.687     247.446  246.846  240.265  257.822
 254.612  258.033  258.238  246.192     262.61   258.278  254.769  261.302
 258.737  256.71   257.43   248.257     257.759  255.152  252.91   262.909
 256.4    258.533  253.001  248.719     260.973  251.786  252.715  264.298
 256.731  257.348  255.837  246.509  …  260.679  262.494  255.798  264.574
 256.381  253.905  259.629  248.907     259.322  259.635  254.254  261.54
 259.127  262.648  262.682  252.241     265.301  259.722  254.632  267.244

Of course, Julia has this one built-in already (it calls BLAS):

In [18]:
C = A * B

1024×1024 Matrix{Float64}:
 259.863  254.538  255.66   246.242  …  261.408  255.862  253.906  260.033
 263.178  264.622  266.778  252.049     268.632  263.909  257.352  271.766
 258.756  254.056  252.085  247.739     258.581  256.31   248.666  260.403
 269.797  265.681  261.253  255.096     272.29   257.799  260.155  274.929
 255.627  251.289  257.524  243.523     256.933  254.892  250.325  261.133
 251.389  252.676  253.779  241.805  …  253.826  251.734  248.212  260.028
 244.499  246.652  245.788  237.687     247.446  246.846  240.265  257.822
 254.612  258.033  258.238  246.192     262.61   258.278  254.769  261.302
 258.737  256.71   257.43   248.257     257.759  255.152  252.91   262.909
 256.4    258.533  253.001  248.719     260.973  251.786  252.715  264.298
 256.731  257.348  255.837  246.509  …  260.679  262.494  255.798  264.574
 256.381  253.905  259.629  248.907     259.322  259.635  254.254  261.54
 259.127  262.648  262.682  252.241     265.301  259.722  254.632  267.244

Our implementation performs quite a bit worse than Julia's (OpenBLAS), but that's OK - we haven't really optimized it at all, since this is just a tutorial:

In [19]:
@benchmark MatrixMultiplication!(C, A, B)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 6.076 s (0.00% GC) to evaluate,
 with a memory estimate of 0 bytes, over 0 allocations.

In [20]:
@benchmark A * B

BenchmarkTools.Trial: 418 samples with 1 evaluation per sample.
 Range (min … max):   5.805 ms … 62.291 ms  ┊ GC (min … max): 0.00% … 1.13%
 Time  (median):      7.585 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   12.023 ms ± 10.578 ms  ┊ GC (mean ± σ):  0.85% ± 3.42%

  ▇▆█▃▂▂                                                       
  ██████▅▆▆▆▇▇▇█▆▆▅▅▅▁▄▄▄▁▁▁▁▁▁▅▄▅▅▄▄▆█▁▅▇▆▆▆▁▅▄▁▁▆▁▁▇▁▁▁▁▄▁▄ ▇
  5.81 ms      Histogram: log(frequency) by time      51.1 ms <

 Memory estimate: 8.00 MiB, allocs estimate: 2.

Ok, let's now implement matrix multiplication on the GPU:

In [21]:
# We need first to move A and B matrices to the GPU and define a new DC zero'd matrix on the GPU
DA = CuArray(A)
DB = CuArray(B)
DC = CUDA.zeros(size(A))

DA

1024×1024 CuArray{Float64, 2, CUDA.DeviceMemory}:
 0.660089   0.910209  0.547063  …  0.964938    0.609727   0.677683
 0.326237   0.544675  0.652396     0.185658    0.455686   0.992389
 0.711058   0.152011  0.392105     0.787188    0.487585   0.804315
 0.281303   0.46956   0.137244     0.24401     0.186616   0.558793
 0.825143   0.204056  0.292768     0.905266    0.908755   0.777511
 0.874345   0.225637  0.822056  …  0.686041    0.0494679  0.0804205
 0.602205   0.351695  0.359766     0.161786    0.724337   0.28782
 0.785481   0.703428  0.904731     0.919084    0.24644    0.548605
 0.505204   0.255183  0.161325     0.780454    0.0791342  0.815757
 0.513395   0.700414  0.850081     0.440594    0.760314   0.385778
 0.72773    0.453     0.494087  …  0.98283     0.48709    0.856695
 0.560429   0.139243  0.598645     0.260218    0.471614   0.286441
 0.127678   0.177489  0.296763     0.836852    0.602944   0.708262
 ⋮                              ⋱                         
 0.367329   0.656017

In the same way, here we can multiply the `DA` matrix by `DB` matrix using the `*` operator (which forwards the call to CUBLAS), thanks again to Julia's multiple dispatch:

In [22]:
DC = DA * DB

1024×1024 CuArray{Float64, 2, CUDA.DeviceMemory}:
 259.863  254.538  255.66   246.242  …  261.408  255.862  253.906  260.033
 263.178  264.622  266.778  252.049     268.632  263.909  257.352  271.766
 258.756  254.056  252.085  247.739     258.581  256.31   248.666  260.403
 269.797  265.681  261.253  255.096     272.29   257.799  260.155  274.929
 255.627  251.289  257.524  243.523     256.933  254.892  250.325  261.133
 251.389  252.676  253.779  241.805  …  253.826  251.734  248.212  260.028
 244.499  246.652  245.788  237.687     247.446  246.846  240.265  257.822
 254.612  258.033  258.238  246.192     262.61   258.278  254.769  261.302
 258.737  256.71   257.43   248.257     257.759  255.152  252.91   262.909
 256.4    258.533  253.001  248.719     260.973  251.786  252.715  264.298
 256.731  257.348  255.837  246.509  …  260.679  262.494  255.798  264.574
 256.381  253.905  259.629  248.907     259.322  259.635  254.254  261.54
 259.127  262.648  262.682  252.241     265.301  25

In [23]:
function MatrixMultiplication_cuda!(C, A, B)
    # Calculate the global row and column indices
    row = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    col = (blockIdx().y - 1) * blockDim().y + threadIdx().y

    # Create a 0 of the same type as C's element type (for type stability)
    sum = zero(eltype(C))

    if row <= size(A, 1) && col < size(B, 2)
        for i in 1:size(A, 2)
            # @inbounds disables bounds checking for array accesses, to improve performance
            # Note that incorrect usage can result in segfaults/memory faults/wrong results
            @inbounds sum += A[row, i] * B[i, col]
        end
        C[row, col] = sum
    end

    return
end

MatrixMultiplication_cuda! (generic function with 1 method)

In [24]:
# Split blocks up into 32x32 tiles
@cuda threads=(32, 32) blocks=(matrix_size ÷ 32, matrix_size ÷ 32) MatrixMultiplication_cuda!(DC, DA, DB)

DC

1024×1024 CuArray{Float64, 2, CUDA.DeviceMemory}:
 259.863  254.538  255.66   246.242  …  261.408  255.862  253.906  260.033
 263.178  264.622  266.778  252.049     268.632  263.909  257.352  271.766
 258.756  254.056  252.085  247.739     258.581  256.31   248.666  260.403
 269.797  265.681  261.253  255.096     272.29   257.799  260.155  274.929
 255.627  251.289  257.524  243.523     256.933  254.892  250.325  261.133
 251.389  252.676  253.779  241.805  …  253.826  251.734  248.212  260.028
 244.499  246.652  245.788  237.687     247.446  246.846  240.265  257.822
 254.612  258.033  258.238  246.192     262.61   258.278  254.769  261.302
 258.737  256.71   257.43   248.257     257.759  255.152  252.91   262.909
 256.4    258.533  253.001  248.719     260.973  251.786  252.715  264.298
 256.731  257.348  255.837  246.509  …  260.679  262.494  255.798  264.574
 256.381  253.905  259.629  248.907     259.322  259.635  254.254  261.54
 259.127  262.648  262.682  252.241     265.301  25

And as we'd expect, the optimized CUBLAS implementation is far faster than ours:

In [25]:
@benchmark DA * DB

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   30.449 μs …  11.082 ms  ┊ GC (min … max): 0.00% … 16.09%
 Time  (median):     133.780 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   129.053 μs ± 156.192 μs  ┊ GC (mean ± σ):  0.27% ±  0.22%

  ▄                                                      ▁▂▃▆█▅ ▁
  █▇█▆▅▄▄▁▃▁▁▃▁▁▁▁▁▃▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▆██████ █
  30.4 μs       Histogram: log(frequency) by time        137 μs <

 Memory estimate: 1.62 KiB, allocs estimate: 71.

In [26]:
@benchmark CUDA.@sync @cuda threads=(32, 32) blocks=(matrix_size ÷ 32, matrix_size ÷ 32) MatrixMultiplication_cuda!(DC, DA, DB)

BenchmarkTools.Trial: 4146 samples with 1 evaluation per sample.
 Range (min … max):  1.194 ms … 1.561 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.204 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.204 ms ± 6.521 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                      ▁▃▂▃▃▆▅▅▆▆▆█▆▅▆▅▄▄▄▃▁▁                 
  ▂▁▂▂▂▂▃▃▃▃▃▃▄▅▅▅▆▇▇▇████████████████████████▇▆▆▅▆▄▃▄▃▃▃▃▃ ▅
  1.19 ms        Histogram: frequency by time       1.21 ms <

 Memory estimate: 1.27 KiB, allocs estimate: 36.

Ouch! Why exactly is our kernel implementation slower?

The answer is that this is only the naive implementation of matrix multiplication - if you did the same in CUDA C++, you'd get similarly bad performance. The performant implementation relies on tiling, where the matrix is divided into smaller submatrices (tiles) that fit more effectively within the GPU’s memory hierarchy, including shared memory and cache. Additionally, optimized kernels would use shared memory and WMMA instructions to greatly improve data locality and reduce bandwidth needs (both of which can be easily used in Julia).

In an optimized implementation, each thread block on the GPU handles a specific tile of the output matrix, loading portions of the input tiles into shared memory to reduce the repeated global memory access. This approach enables a higher level of parallelism by allowing multiple tiles to be processed concurrently across the GPU cores, without bottlenecking on memory transfers.

For the purpose of this tutorial, we will not be implementing these optimizations, but do know that they are as easy (or easier) to use in Julia compared to CUDA C++.

# KernelAbstractions

Now that we know how to write vendor-specific kernels, let's explore the naive matrix multiplication example using KernelAbstractions.jl:

In [27]:
import Pkg
Pkg.add("KernelAbstractions")

   Resolving package versions...
    Updating `/global/u1/o/omairyrm/.julia/environments/v1.10/Project.toml`
  [63c18a36] + KernelAbstractions v0.9.34
  No Changes to `/global/u1/o/omairyrm/.julia/environments/v1.10/Manifest.toml`


We'll also load the Random standard library to assist with certain random array initializations:

In [28]:
using KernelAbstractions
using Random

Implementing a kernel with KernelAbstractions is very similar to implementing a kernel with CUDA.jl. The primary differences include annotating a kernel function with `@kernel`, and doing thread indexing using `@index` (which efficiently abstracts away the index calculations we were previously doing). Otherwise, most things are the same:

In [29]:
@kernel function MatrixMultiplication_kernel!(C, A, B)
    # Global index of each thread across multiple blocks in both x and y dimensions of the grid
    row, col = @index(Global, NTuple)

    # Everything else is the same!
    sum = zero(eltype(C))

    if row <= size(A, 1) && col <= size(B, 2)
        for i = 1:size(A, 2)
             @inbounds sum += A[row, i] * B[i, col]
        end
        @inbounds C[row, col] = sum
     end
end

MatrixMultiplication_kernel! (generic function with 4 methods)

One key difference between KernelAbstractions and CUDA is that, because KernelAbstractions is portable, we need to select the CUDA "backend" when we compile our kernel (AMDGPU, Apple, and Intel are also supported). Most operations take the backend as the first argument, to allow Julia's multiple dispatch to redirect calls to the correct implementation. Additionally, KernelAbstractions separate the compilation and kernel launch stages, and provides configurations for each step to optimize further.

In [30]:
# Select the CUDA backend
Backend = CUDA.CUDABackend()

# Use KernelAbstractions's APIs to allocate GPU matrices DA, DB, and DC
matrix_size = 2048
T = Float64
DA = rand!(allocate(Backend, T, matrix_size, matrix_size))
DB = rand!(allocate(Backend, T, matrix_size, matrix_size))
DC = KernelAbstractions.zeros(Backend, T, matrix_size, matrix_size)

# Compile the kernel
# We'll statically assign the workgroup (AKA block) size to allow for additional compile-time optimizations
workgroupsize = (32, 32)
kernel! = MatrixMultiplication_kernel!(Backend, workgroupsize)

# Launch the kernel with our GPU matrices as inputs
kernel!(DC, DA, DB, ndrange=(size(DC)))

# Explicitly wait for the kernel to complete
KernelAbstractions.synchronize(Backend)

# Are our results what we'd expect to see (compared to CUBLAS)?
isapprox(DC, DA * DB)

true

In [31]:
@benchmark begin
    kernel!(DC, DA, DB, ndrange=size(DC))
    KernelAbstractions.synchronize(Backend)
end

BenchmarkTools.Trial: 359 samples with 1 evaluation per sample.
 Range (min … max):  13.844 ms … 14.196 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     13.917 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   13.918 ms ± 29.861 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                         ▁ ▅▁▅▇▄▇▄▅▄▃▅█▂▃▂ ▁▁                  
  ▃▃▁▃▃▁▁▁▃▃▃▄▃▃▃▃▃▆▆▅▆▇██▇██████████████████▆▆▆▆▃▆▆▃▃▁▃▃▁▃▃▄ ▄
  13.8 ms         Histogram: frequency by time          14 ms <

 Memory estimate: 1.92 KiB, allocs estimate: 64.